In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_11_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 12 ###

### cell 12 (further optimized for cudf)

def bar_chart_multiple_choice_24(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    response_counts.index.name = ''
    # write out to CSV directly from GPU Series (no to_frame conversion)
    response_counts.to_csv(
        f'/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/{title}.csv',
        index=True
    )
    # compute the 1.2× scaled max on GPU without using iloc
    tmp_cmp = response_counts.head(num_choices).mul(1.2).max()


def count_then_return_percent_24(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    total = counts.sum()
    return counts.mul(100).div(total).round(1)

# filter once using cudf .loc
USA_responses_df_2022 = responses_df_2022_cell10.loc[
    responses_df_2022_cell10['In which country do you currently reside?'] == 'United States of America'
]
question_of_interest_cell24 = 'Are you currently a student? (high school, university, or graduate)'
percentages_cell24 = (
    count_then_return_percent_24(USA_responses_df_2022, question_of_interest_cell24)
    .sort_index()
)
title_for_chart_cell24 = 'Students status for Kaggle Survey participants from the USA'
title_for_y_axis_cell24 = '% of respondents'
bar_chart_multiple_choice_24(
    response_counts=percentages_cell24,
    title=title_for_chart_cell24,
    y_axis_title=title_for_y_axis_cell24,
    orientation=orientation_for_chart
)

# repeat for India
India_responses_df_2022 = responses_df_2022_cell10.loc[
    responses_df_2022_cell10['In which country do you currently reside?'] == 'India'
]
percentages_cell24 = (
    count_then_return_percent_24(India_responses_df_2022, question_of_interest_cell24)
    .sort_index()
)
title_for_chart_cell24 = 'Students status for Kaggle Survey participants from India'
title_for_y_axis_cell24 = '% of respondents'
# match original behavior by printing summary and returning None
percentages_cell24.info()